# Smoke Detection Prototype — Retinex + YOLOv9 + ByteTrack + Smoke Validation
**Garay & Rañola**

Upload a video, pick a model, and the prototype runs the full Config 3 pipeline on it:
Retinex enhancement → YOLOv9 detection → ByteTrack → Section 3.4 smoke validation.
It returns an annotated video, a verdict, and a timeline of detections.

The Retinex functions, tracker, and validation checks are copied from the
thesis notebook, so the prototype behaves the same way as the evaluated pipeline.

**How to use**
1. Runtime → Change runtime type → **GPU**.
2. Run all cells top to bottom (Runtime → Run all).
3. Open the **public Gradio link** printed by the last UI cell (or use the inline app).
4. Upload a video, choose a model, click **Run detection**.

**Box colors in the output video**
- Yellow (thin): raw YOLOv9 detection, with confidence
- Blue: tracked region (ByteTrack ID and persistence score *P*)
- Red (thick): **confirmed smoke** — passed persistence + all Section 3.4 checks

## 1. Setup

In [ ]:
import os, sys, torch
from google.colab import drive

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

CUDA available: True
GPU: NVIDIA L4
Mounted at /content/drive


In [ ]:
if not os.path.exists('/content/yolov9'):
    !git clone -q https://github.com/WongKinYiu/yolov9 /content/yolov9
%cd /content/yolov9
!pip install -q -r requirements.txt
!pip install -q gradio scipy
sys.path.insert(0, '/content/yolov9')

/content/yolov9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 105.2 MB/s eta 0:00:00


In [ ]:
import re

def patch_all_torch_loads(repo_dir):
    fixed_files = 0
    for root, _, files in os.walk(repo_dir):
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, 'r') as f:
                content = f.read()

            # Match ANY torch.load(...) call that doesn't already set weights_only
            pattern = re.compile(r"torch\.load\(([^()]*(?:\([^()]*\)[^()]*)*)\)")

            def add_weights_only(match):
                inner = match.group(1)
                if 'weights_only' in inner:
                    return match.group(0)
                return f'torch.load({inner}, weights_only=False)'

            new_content = pattern.sub(add_weights_only, content)

            if new_content != content:
                with open(fpath, 'w') as f:
                    f.write(new_content)
                fixed_files += 1

    print(f'Patched {fixed_files} files in {repo_dir}')

patch_all_torch_loads('/content/yolov9')

Patched 10 files in /content/yolov9


## 2. Locate Trained Weights

Searches the Drive folders below for each model's `best.pt`. Add your own folder to
`WEIGHT_SEARCH_DIRS`, or point a model straight at a file with `WEIGHT_OVERRIDES`.

In [ ]:
import json

DRIVE_THESIS_DIR = '/content/drive/MyDrive/THESIZZZ'

WEIGHT_SEARCH_DIRS = [
    f'{DRIVE_THESIS_DIR}/Results',
    f'{DRIVE_THESIS_DIR}/PAPERZ/FINAL FORMS - T3/GR_THESIS3_SUBMISSION/Thesis Proof of Concept/Dataset/Results',
]
WEIGHT_OVERRIDES = {}      # e.g. {'amsrcr': '/content/drive/MyDrive/somewhere/best.pt'}

MODEL_LABELS = {
    'baseline': 'Baseline (no enhancement)',
    'ssr'     : 'SSR + YOLOv9',
    'msr'     : 'MSR + YOLOv9',
    'msrcr'   : 'MSRCR + YOLOv9',
    'amsrcr'  : 'AMSRCR + YOLOv9',
}

# Where outputs are saved (local, plus a copy on Drive)
OUT_DIR       = '/content/prototype_outputs'
DRIVE_OUT_DIR = f'{DRIVE_THESIS_DIR}/Prototype_Outputs'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)


def find_weights(key):
    if key in WEIGHT_OVERRIDES and os.path.exists(WEIGHT_OVERRIDES[key]):
        return WEIGHT_OVERRIDES[key]
    folders = [f'smoke_{key}_weights'] + (['smoke_retinex_weights'] if key == 'amsrcr' else [])
    for base in WEIGHT_SEARCH_DIRS:
        for folder in folders:
            path = f'{base}/{folder}/best.pt'
            if os.path.exists(path):
                return path
    return None


WEIGHT_PATHS = {k: find_weights(k) for k in MODEL_LABELS}
AVAILABLE_MODELS = [k for k, p in WEIGHT_PATHS.items() if p]

assert AVAILABLE_MODELS, 'No trained weights found -- check WEIGHT_SEARCH_DIRS.'

# Selected variant from the thesis notebook (detection-based selection) -> default model
SELECTED = None
for base in WEIGHT_SEARCH_DIRS:
    path = f'{base}/selected_variant.json'
    if os.path.exists(path):
        with open(path) as f:
            SELECTED = json.load(f).get('chosen')
        break
if SELECTED in AVAILABLE_MODELS:
    MODEL_LABELS[SELECTED] += '  (selected)'
DEFAULT_MODEL = SELECTED if SELECTED in AVAILABLE_MODELS else AVAILABLE_MODELS[0]

for k, p in WEIGHT_PATHS.items():
    print(f'  {"✓" if p else "✗"} {MODEL_LABELS[k]:<34} {p or "not found"}')
print(f'\nDefault model: {MODEL_LABELS[DEFAULT_MODEL]}')

  ✓ Baseline (no enhancement)          /content/drive/MyDrive/THESIZZZ/Results/smoke_baseline_weights/best.pt
  ✓ SSR + YOLOv9                       /content/drive/MyDrive/THESIZZZ/Results/smoke_ssr_weights/best.pt
  ✓ MSR + YOLOv9                       /content/drive/MyDrive/THESIZZZ/Results/smoke_msr_weights/best.pt
  ✓ MSRCR + YOLOv9  (selected)         /content/drive/MyDrive/THESIZZZ/Results/smoke_msrcr_weights/best.pt
  ✓ AMSRCR + YOLOv9                    /content/drive/MyDrive/THESIZZZ/Results/smoke_amsrcr_weights/best.pt

Default model: MSRCR + YOLOv9  (selected)


## 3. Retinex Module (same as thesis Section 4)

In [ ]:
import cv2
import numpy as np

def single_scale_retinex(image: np.ndarray, sigma: float) -> np.ndarray:
    """Single-Scale Retinex"""
    gaussian = cv2.GaussianBlur(image, (0, 0), sigma)
    retinex  = np.log1p(image.astype(np.float32)) - np.log1p(gaussian.astype(np.float32))
    return retinex


def multi_scale_retinex(image: np.ndarray, sigmas=(15, 80, 200)) -> np.ndarray:
    """Multi-Scale Retinex"""
    msr = np.zeros_like(image, dtype=np.float32)
    for sigma in sigmas:
        msr += single_scale_retinex(image, sigma)
    msr /= len(sigmas)
    return msr


def ssr_enhance(image_bgr: np.ndarray, sigma: float = 15) -> np.ndarray:
    """Single-Scale Retinex enhancement."""
    img    = image_bgr.astype(np.float32) + 1.0
    result = np.zeros_like(img)
    for c in range(3):
        ch = single_scale_retinex(img[:, :, c], sigma)
        result[:, :, c] = (ch - ch.min()) / (ch.max() - ch.min() + 1e-6)
    return np.clip(result * 255, 0, 255).astype(np.uint8)


def msr_enhance(image_bgr: np.ndarray, sigmas=(15, 80, 200)) -> np.ndarray:
    """Multi-Scale Retinex enhancement no color restoration."""
    img    = image_bgr.astype(np.float32) + 1.0
    result = np.zeros_like(img)
    for c in range(3):
        ch = multi_scale_retinex(img[:, :, c], sigmas)
        result[:, :, c] = (ch - ch.min()) / (ch.max() - ch.min() + 1e-6)
    return np.clip(result * 255, 0, 255).astype(np.uint8)


def retinex_enhance(image_bgr: np.ndarray, sigmas=(15, 80, 200)) -> np.ndarray:
    """
    MSRCR pipeline (Section 3.5.1):
      1. Convert to float
      2. Apply MSR per channel
      3. Color restoration
      4. Normalize to [0, 255] uint8
    """
    img = image_bgr.astype(np.float32) + 1.0

    msr_channels = [multi_scale_retinex(img[:, :, c], sigmas) for c in range(3)]
    msr = np.stack(msr_channels, axis=2)

    total         = np.sum(img, axis=2, keepdims=True)
    color_restore = np.log1p(125.0 * img / (total + 1e-6))
    msr           = msr * color_restore

    for c in range(3):
        ch = msr[:, :, c]
        msr[:, :, c] = (ch - ch.min()) / (ch.max() - ch.min() + 1e-6)

    return np.clip(msr * 255, 0, 255).astype(np.uint8)


def automated_msrcr(image_bgr: np.ndarray, sigmas=(15, 80, 200),
                     alpha: float = 125, beta: float = 46) -> np.ndarray:
    """
    AMSRCR — per-channel MSR with color restoration + automatic gain/offset
    determined from the mean/std of each channel (no fixed normalization).
    """
    img_f  = image_bgr.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    total  = np.sum(img_f, axis=2) + 1e-6

    for c in range(3):
        ch        = img_f[:, :, c]
        msr       = multi_scale_retinex(ch, sigmas)
        cr        = beta * (np.log(alpha * ch) - np.log(total))
        msrcr_ch  = msr * cr
        mean, std = msrcr_ch.mean(), msrcr_ch.std() + 1e-6
        result[:, :, c] = np.clip((msrcr_ch - mean) / std * 64 + 128, 0, 255)

    return np.uint8(result)


VARIANT_FUNCS = {
    'ssr'    : ssr_enhance,
    'msr'    : msr_enhance,
    'msrcr'  : retinex_enhance,
    'amsrcr' : automated_msrcr,
}

print('Retinex module loaded: SSR, MSR, MSRCR, AMSRCR ready.')

Retinex module loaded: SSR, MSR, MSRCR, AMSRCR ready.


## 4. Detection, Tracking & Smoke Validation (same as thesis Sections 9, 10, 12)

In [ ]:
import cv2, time, gc
import numpy as np
from models.common import DetectMultiBackend
from utils.general import non_max_suppression
from utils.torch_utils import select_device

IMG_SIZE = 640
DEVICE   = '0' if torch.cuda.is_available() else 'cpu'


def letterbox(img, new_shape=640, color=(114, 114, 114)):
    shape = img.shape[:2]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = (int(round(shape[1] * r)), int(round(shape[0] * r)))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]
    dw /= 2
    dh /= 2
    if shape[::-1] != new_unpad:
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return img, r, (dw, dh)


def estimate_background(frame_buffer):
    return np.mean(np.stack(frame_buffer, axis=0), axis=0).astype(np.uint8)


def iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter / (area1 + area2 - inter + 1e-6)


class SmokeTrack:
    _next_id = 1
    def __init__(self, box, score):
        self.id = SmokeTrack._next_id
        SmokeTrack._next_id += 1
        self.box, self.score = box, score
        self.hits, self.misses, self.total_frames = 1, 0, 1
        self.history = [box]

    def update(self, box, score):
        self.box, self.score = box, score
        self.hits += 1
        self.misses = 0
        self.total_frames += 1
        self.history.append(box)

    def mark_missed(self):
        self.misses += 1
        self.total_frames += 1

    @property
    def persistence_score(self):
        return self.hits / max(self.total_frames, 1)

    @property
    def area_diffusion(self):
        if len(self.history) < 2:
            return 0.0
        b_prev, b_curr = self.history[-2], self.history[-1]
        area_prev = (b_prev[2]-b_prev[0]) * (b_prev[3]-b_prev[1])
        area_curr = (b_curr[2]-b_curr[0]) * (b_curr[3]-b_curr[1])
        return area_curr - area_prev


class ByteTracker:
    def __init__(self, high_thresh=0.5, low_thresh=0.1, iou_thresh=0.3,
                 max_misses=5, persistence_thresh=0.4):
        self.high_thresh, self.low_thresh = high_thresh, low_thresh
        self.iou_thresh, self.max_misses = iou_thresh, max_misses
        self.persistence_thresh = persistence_thresh
        self.tracks = []

    def _match(self, tracks, detections):
        if not tracks or not detections:
            return [], list(range(len(tracks))), list(range(len(detections)))
        from scipy.optimize import linear_sum_assignment
        cost = np.array([[1 - iou(t.box, d['box']) for d in detections] for t in tracks])
        ri, ci = linear_sum_assignment(cost)
        matched, unmatched_t, unmatched_d = [], [], []
        for r, c in zip(ri, ci):
            if cost[r, c] < 1 - self.iou_thresh:
                matched.append((r, c))
            else:
                unmatched_t.append(r); unmatched_d.append(c)
        matched_t = [m[0] for m in matched]; matched_d = [m[1] for m in matched]
        unmatched_t += [i for i in range(len(tracks)) if i not in matched_t]
        unmatched_d += [i for i in range(len(detections)) if i not in matched_d]
        return matched, unmatched_t, unmatched_d

    def update(self, detections):
        high = [d for d in detections if d['score'] >= self.high_thresh]
        low  = [d for d in detections if self.low_thresh <= d['score'] < self.high_thresh]
        matched, unmatched_t, unmatched_d = self._match(self.tracks, high)
        for ti, di in matched:
            self.tracks[ti].update(high[di]['box'], high[di]['score'])
        remaining = [self.tracks[i] for i in unmatched_t]
        matched2, unmatched_t2, _ = self._match(remaining, low)
        for ti, di in matched2:
            remaining[ti].update(low[di]['box'], low[di]['score'])
        for i in [unmatched_t[j] for j in unmatched_t2]:
            self.tracks[i].mark_missed()
        for di in unmatched_d:
            self.tracks.append(SmokeTrack(high[di]['box'], high[di]['score']))
        self.tracks = [t for t in self.tracks if t.misses <= self.max_misses]
        return [t for t in self.tracks if t.persistence_score >= self.persistence_thresh]


def intensity_variance(image_gray, x1, y1, x2, y2):
    roi = image_gray[int(y1):int(y2), int(x1):int(x2)]
    if roi.size == 0:
        return 0.0
    return float(np.mean((roi.astype(float) - roi.mean()) ** 2))


def smoke_density(background_gray, current_gray, x1, y1, x2, y2):
    roi_bg  = background_gray[int(y1):int(y2), int(x1):int(x2)].astype(float)
    roi_cur = current_gray[int(y1):int(y2), int(x1):int(x2)].astype(float)
    if roi_bg.size == 0:
        return 0.0
    return float(np.mean(roi_bg - roi_cur))


def evaluate_smoke_track(track, frame_gray, background_gray, thresholds=None):
    if thresholds is None:
        thresholds = {
            'intensity_variation_min': 50.0,
            'smoke_density_min': 5.0,
            'temporal_diffusion_min': 0.0,
            'temporal_persistence_min': 0.4,
        }
    x1, y1, x2, y2 = track.box
    metrics = {
        'intensity_variation': intensity_variance(frame_gray, x1, y1, x2, y2),
        'smoke_density': smoke_density(background_gray, frame_gray, x1, y1, x2, y2),
        'temporal_diffusion': track.area_diffusion,
        'temporal_persistence': track.persistence_score,
    }
    confirmed = (
        metrics['intensity_variation'] >= thresholds['intensity_variation_min'] and
        metrics['smoke_density'] >= thresholds['smoke_density_min'] and
        metrics['temporal_diffusion'] >= thresholds['temporal_diffusion_min'] and
        metrics['temporal_persistence'] >= thresholds['temporal_persistence_min']
    )
    return confirmed, metrics


_model_cache = {}

def get_model(key):
    """Loads a model once and keeps it in memory for later runs."""
    if key not in _model_cache:
        device = select_device(DEVICE)
        model = DetectMultiBackend(WEIGHT_PATHS[key], device=device, dnn=False)
        model.eval()
        _model_cache[key] = (model, device)
    return _model_cache[key]

print('Pipeline ready.')

/content/yolov9/utils/general.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


Pipeline ready.


## 5. Prototype Function

`run_prototype()` processes one video and returns the annotated output video, a summary,
and a timeline figure. Like the thesis evaluation, it runs the pipeline on every
`frame_skip`-th frame. The frames in between are still written to the output video and
show the most recent result, so playback stays smooth.

In [ ]:
import subprocess, shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

YELLOW, BLUE, RED, GREEN, WHITE = (0, 220, 255), (255, 140, 0), (0, 0, 255), (0, 190, 0), (255, 255, 255)


def _enhanced_display(model_in, ratio, pad, w, h):
    """Undo the letterbox on the model input so it lines up with the original frame."""
    nw, nh = int(round(w * ratio)), int(round(h * ratio))
    top, left = int(round(pad[1] - 0.1)), int(round(pad[0] - 0.1))
    crop = model_in[top:top + nh, left:left + nw]
    return cv2.resize(crop, (w, h), interpolation=cv2.INTER_LINEAR)


def _draw_boxes(img, state, thick):
    h, w = img.shape[:2]
    clip = lambda b: (int(max(0, min(b[0], w - 1))), int(max(0, min(b[1], h - 1))),
                      int(max(0, min(b[2], w - 1))), int(max(0, min(b[3], h - 1))))
    font, fs = cv2.FONT_HERSHEY_SIMPLEX, 0.45 * thick
    confirmed_ids = {tid for tid, _ in state['confirmed']}

    for d in state['dets']:
        x1, y1, x2, y2 = clip(d['box'])
        cv2.rectangle(img, (x1, y1), (x2, y2), YELLOW, max(1, thick - 1))
        cv2.putText(img, f"{d['score']:.2f}", (x1, min(h - 4, y2 + int(14 * thick))), font, fs * 0.8, YELLOW, max(1, thick - 1))
    for tid, box, p in state['active']:
        if tid in confirmed_ids:
            continue
        x1, y1, x2, y2 = clip(box)
        cv2.rectangle(img, (x1, y1), (x2, y2), BLUE, thick)
        cv2.putText(img, f'ID{tid} P={p:.2f}', (x1, max(12, y1 - 5)), font, fs, BLUE, thick)
    for tid, box in state['confirmed']:
        x1, y1, x2, y2 = clip(box)
        cv2.rectangle(img, (x1, y1), (x2, y2), RED, thick + 2)
        cv2.putText(img, f'SMOKE ID{tid}', (x1, max(12, y1 - 5)), font, fs * 1.1, RED, thick + 1)
    return img


def _banner(img, text_left, smoke):
    """Black bar across the top: info on the left, status on the right (text shrinks to fit)."""
    h, w = img.shape[:2]
    bar_h = max(28, h // 16)
    cv2.rectangle(img, (0, 0), (w, bar_h), (0, 0, 0), -1)
    font = cv2.FONT_HERSHEY_SIMPLEX
    status, color = ('SMOKE CONFIRMED', RED) if smoke else ('Monitoring', GREEN)
    fs_status = bar_h / 38
    (sw, _), _ = cv2.getTextSize(status, font, fs_status, 2)
    cv2.putText(img, status, (w - sw - 10, int(bar_h * 0.72)), font, fs_status, color, 2)
    fs = bar_h / 42
    while fs > 0.3 and cv2.getTextSize(text_left, font, fs, 1)[0][0] > w - sw - 30:
        fs -= 0.05
    cv2.putText(img, text_left, (10, int(bar_h * 0.68)), font, fs, WHITE, 1)
    return img


def _panel_label(img, text):
    """Small caption in the bottom-left corner of a panel."""
    h, w = img.shape[:2]
    fs = max(0.4, min(w, h) / 900)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, 1)
    cv2.rectangle(img, (0, h - th - 14), (tw + 16, h), (0, 0, 0), -1)
    cv2.putText(img, text, (8, h - 7), cv2.FONT_HERSHEY_SIMPLEX, fs, WHITE, 1)
    return img


def _to_browser_mp4(src, dst):
    """Re-encode to H.264 so the video plays in the browser / Gradio."""
    r = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src, '-c:v', 'libx264',
                        '-pix_fmt', 'yuv420p', '-movflags', '+faststart', dst], capture_output=True)
    if r.returncode == 0 and os.path.exists(dst):
        os.remove(src)
        return dst
    return src


def run_prototype(video_path, model_key=None, view='Side by side', frame_skip=5,
                  conf_thresh=0.25, iou_thresh=0.45, max_seconds=60, progress_cb=None):
    model_key = model_key or DEFAULT_MODEL
    model, device = get_model(model_key)
    enhance_fn = VARIANT_FUNCS.get(model_key)          # None for the baseline
    if enhance_fn is None and view != 'Original':
        view = 'Original'                               # nothing to show as "enhanced"

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f'Could not open video: {video_path}')
    fps_in = cap.get(cv2.CAP_PROP_FPS)
    fps_in = fps_in if 1 < fps_in <= 120 else 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    limit = int(max_seconds * fps_in) if max_seconds else total_frames
    limit = min(limit, total_frames) if total_frames > 1 else limit
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out_w = w * 2 if view == 'Side by side' else w
    thick = max(1, round(min(w, h) / 320))

    stem = os.path.splitext(os.path.basename(video_path))[0]
    stamp = time.strftime('%Y%m%d_%H%M%S')
    tmp_out = f'{OUT_DIR}/{stem}_{model_key}_{stamp}_raw.mp4'
    final_out = f'{OUT_DIR}/{stem}_{model_key}_{stamp}.mp4'
    writer = cv2.VideoWriter(tmp_out, cv2.VideoWriter_fourcc(*'mp4v'), fps_in, (out_w, h))

    SmokeTrack._next_id = 1
    tracker = ByteTracker(persistence_thresh=0.4)
    bg_buffer, frame_times, timeline = [], [], []
    state = {'dets': [], 'active': [], 'confirmed': [], 'enh': None}
    raw_idx = 0
    first_confirm_s = None

    while raw_idx < limit:
        ret, frame = cap.read()
        if not ret:
            break
        raw_idx += 1

        if raw_idx % frame_skip == 0:
            t0 = time.perf_counter()
            img_r, ratio, (pad_w, pad_h) = letterbox(frame, IMG_SIZE)
            model_in = enhance_fn(img_r) if enhance_fn else img_r
            rgb = cv2.cvtColor(model_in, cv2.COLOR_BGR2RGB)
            tensor = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = non_max_suppression(model(tensor), conf_thresh, iou_thresh, max_det=100)

            dets = []
            for x1, y1, x2, y2, conf, _ in (pred[0].tolist() if pred[0] is not None else []):
                dets.append({'box': [(x1 - pad_w) / ratio, (y1 - pad_h) / ratio,
                                     (x2 - pad_w) / ratio, (y2 - pad_h) / ratio], 'score': conf})

            active = tracker.update(dets)
            frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            bg_buffer.append(frame_gray)
            if len(bg_buffer) > 10:
                bg_buffer.pop(0)
            background_gray = estimate_background(bg_buffer) if len(bg_buffer) > 1 else frame_gray
            confirmed = [t for t in active if evaluate_smoke_track(t, frame_gray, background_gray)[0]]
            frame_times.append(time.perf_counter() - t0)

            t_s = raw_idx / fps_in
            if confirmed and first_confirm_s is None:
                first_confirm_s = t_s
            timeline.append((t_s, len(dets), len(confirmed)))
            state = {'dets': [dict(d) for d in dets],
                     'active': [(t.id, list(t.box), t.persistence_score) for t in active],
                     'confirmed': [(t.id, list(t.box)) for t in confirmed],
                     'enh': (_enhanced_display(model_in, ratio, (pad_w, pad_h), w, h)
                             if enhance_fn and view != 'Original' else None)}

        # ── draw every frame using the latest pipeline result ──
        fps_now = 1.0 / np.mean(frame_times[-20:]) if frame_times else 0.0
        info = f'{MODEL_LABELS[model_key]} | t={raw_idx / fps_in:5.1f}s | {fps_now:4.1f} FPS'
        smoke = bool(state['confirmed'])
        orig = _draw_boxes(frame.copy(), state, thick)
        if view == 'Original':
            out = _banner(orig, info, smoke)
        else:
            enh = state['enh'] if state['enh'] is not None else frame.copy()
            enh = _draw_boxes(enh.copy(), state, thick)
            if view == 'Enhanced':
                out = _banner(enh, info, smoke)
            else:
                out = _banner(np.hstack([_panel_label(orig, 'Original video'),
                                         _panel_label(enh, f'{model_key.upper()}-enhanced (model input)')]),
                              info, smoke)
        writer.write(out)

        if progress_cb and raw_idx % 10 == 0:
            progress_cb(raw_idx / max(limit, 1))

    cap.release()
    writer.release()
    final_out = _to_browser_mp4(tmp_out, final_out)
    try:
        shutil.copy(final_out, DRIVE_OUT_DIR)
    except Exception as e:
        print('Could not copy to Drive:', e)

    confirmed_frames = sum(1 for _, _, c in timeline if c > 0)
    summary = {
        'video': os.path.basename(video_path),
        'model': MODEL_LABELS[model_key],
        'verdict': 'SMOKE DETECTED' if confirmed_frames else 'NO SMOKE CONFIRMED',
        'seconds_analyzed': round(raw_idx / fps_in, 1),
        'frames_processed': len(timeline),
        'raw_detections': sum(d for _, d, _ in timeline),
        'frames_with_confirmed_smoke': confirmed_frames,
        'first_confirmation_s': None if first_confirm_s is None else round(first_confirm_s, 1),
        'avg_pipeline_fps': round(1.0 / np.mean(frame_times), 2) if frame_times else 0.0,
        'output_video': final_out,
    }

    fig, ax = plt.subplots(figsize=(8, 2.8))
    if timeline:
        ts, raw, conf = zip(*timeline)
        ax.plot(ts, raw, color='#f2b134', label='Raw detections')
        ax.fill_between(ts, conf, step='mid', color='#e15759', alpha=0.6, label='Confirmed smoke')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Count')
    ax.set_title('Detections over time'); ax.legend(loc='upper right'); ax.grid(alpha=0.3)
    fig.tight_layout()
    return final_out, summary, fig

print('run_prototype() ready.')

run_prototype() ready.


## 6. Prototype App (Gradio)

Run this cell, then open the **public link** (`https://....gradio.live`) it prints, or use
the app inline below the cell. The cell keeps running while the app is up; stop it with
the ■ button when you're done.

In [ ]:
import gradio as gr


def ui_run(video, model_key, view, frame_skip, conf_thresh, max_seconds, progress=gr.Progress()):
    if video is None:
        raise gr.Error('Please upload a video first.')
    progress(0, desc='Loading model...')
    out_path, s, fig = run_prototype(video, model_key, view, int(frame_skip), float(conf_thresh),
                                     max_seconds=int(max_seconds) or None,
                                     progress_cb=lambda f: progress(f, desc='Processing video...'))
    color = '#d62728' if s['verdict'] == 'SMOKE DETECTED' else '#2ca02c'
    first = f"{s['first_confirmation_s']} s" if s['first_confirmation_s'] is not None else '—'
    report = (
        f"<h2 style='color:{color};margin:0'>{s['verdict']}</h2>\n\n"
        f"| | |\n|---|---|\n"
        f"| Model | {s['model']} |\n"
        f"| Seconds analyzed | {s['seconds_analyzed']} |\n"
        f"| Frames processed | {s['frames_processed']} (every {int(frame_skip)} frames) |\n"
        f"| Raw YOLOv9 detections | {s['raw_detections']} |\n"
        f"| Frames with confirmed smoke | {s['frames_with_confirmed_smoke']} |\n"
        f"| First confirmation | {first} |\n"
        f"| Average pipeline FPS | {s['avg_pipeline_fps']} |\n"
    )
    return out_path, report, fig


default_model = DEFAULT_MODEL

with gr.Blocks(title='Retinex Smoke Detection Prototype') as demo:
    gr.Markdown('# Retinex-Based Low-Light Smoke Detection — Prototype\n'
                'Retinex enhancement → YOLOv9 → ByteTrack → mathematical smoke validation. '
                '**Yellow** = raw detection, **blue** = tracked region, **red** = confirmed smoke.')
    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label='Input video', sources=['upload'])
            model_dd = gr.Dropdown(choices=[(MODEL_LABELS[k], k) for k in AVAILABLE_MODELS],
                                   value=default_model, label='Model')
            view_rb  = gr.Radio(['Side by side', 'Original', 'Enhanced'], value='Side by side',
                                label='Output view')
            with gr.Accordion('Advanced settings', open=False):
                skip_sl = gr.Slider(1, 10, value=5, step=1, label='Frame skip (5 = same as thesis evaluation)')
                conf_sl = gr.Slider(0.05, 0.9, value=0.25, step=0.05, label='YOLOv9 confidence threshold')
                max_sl  = gr.Slider(0, 300, value=60, step=10, label='Max seconds to analyze (0 = whole video)')
            run_btn = gr.Button('Run detection', variant='primary')
        with gr.Column(scale=2):
            video_out = gr.Video(label='Annotated output')
            report_md = gr.Markdown()
            timeline_plot = gr.Plot(label='Timeline')

    run_btn.click(ui_run, [video_in, model_dd, view_rb, skip_sl, conf_sl, max_sl],
                  [video_out, report_md, timeline_plot])

demo.queue().launch(share=True, debug=True, allowed_paths=[OUT_DIR, DRIVE_OUT_DIR])

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b99c28636d0d232460.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


YOLO 🚀 v0.1-104-g5b1ea9a Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)

Fusing layers... 
gelan-c summary: 387 layers, 25227859 parameters, 0 gradients, 101.8 GFLOPs


## 7. Without the App (optional)

If the Gradio link doesn't work, you can run the prototype directly on a video file.
The annotated video is saved to `OUT_DIR` and copied to `THESIZZZ/Prototype_Outputs` on Drive.

In [ ]:
from IPython.display import Video, display

VIDEO_PATH = '/content/your_video.mp4'     # <- upload a video to /content and put its path here

out_path, summary, fig = run_prototype(VIDEO_PATH, model_key=DEFAULT_MODEL, view='Side by side')
for k, v in summary.items():
    print(f'{k:<28} {v}')
display(fig)
display(Video(out_path, embed=True, width=900))